<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part3_3_TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Part 3.3 - Backbone Fine Tune

ResNet50 on Block 1 for age category classification. Two runs, frozen backbone with head only, then full fine tune.

Compared against notebook 8 (BestModel from scratch) and notebook 10 (autoencoder transfer).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA A100-SXM4-40GB


## 3. Split Block 1 for classification

Block 1 split 80/20 train/val, stratified by age_category to keep class balance. Same approach used on Block 2 in notebook 10.

In [4]:
b1 = pd.read_csv("/content/data_splits/block1.csv")
b1_train, b1_val = train_test_split(
    b1, test_size=0.2, stratify=b1["age_category"], random_state=42
)
b1_train.to_csv("/content/b1_train.csv", index=False)
b1_val.to_csv("/content/b1_val.csv", index=False)
print(f"block1 total {len(b1)}  train {len(b1_train)}  val {len(b1_val)}")
print(b1["age_category"].value_counts())

block1 total 4392  train 3513  val 879
age_category
infant    959
youth     732
mature    616
mid       587
senior    583
child     506
teen      409
Name: count, dtype: int64
